# Redes Neuronales Recurrentes: detección de anomalías en logs de HDFS

En las clases anteriores las entradas tenían tamaño fijo: un vector de características (MLP) o una imagen (CNN). En esta clase pasamos a **secuencias**: listas ordenadas de eventos cuyo largo varía de un ejemplo a otro. Para procesarlas presentamos las **Redes Neuronales Recurrentes (RNN)**, que recorren la secuencia paso a paso y mantienen un **estado oculto** que resume lo visto hasta ese momento.

El problema consiste en clasificar como **normales** o **anómalas** las secuencias de eventos que registra HDFS (*Hadoop Distributed File System*), un sistema de archivos distribuido. Detectar anomalías en los logs permite identificar fallas sin que un operador revise millones de líneas.

## Introducción

### Objetivos

1. **Entender cómo un log de texto se convierte en una secuencia de enteros** y cómo se lleva a tensores de largo fijo (padding y truncado).
2. **Representar eventos categóricos con `nn.Embedding`**, una tabla de vectores que se aprende junto con el resto de la red.
3. **Entender la recurrencia de `nn.RNN`**: la fórmula, los pesos compartidos entre pasos y sus dos salidas.
4. **Entrenar un clasificador de secuencias** con un dataset desbalanceado y evaluarlo con métricas adecuadas.

### Contenido

1. Configuración de bibliotecas y semillas para reproducibilidad.
2. Del log crudo a la secuencia de eventos; análisis exploratorio.
3. Preprocesamiento: padding y truncado.
4. `Dataset` y `DataLoader`.
5. `nn.Embedding`.
6. `nn.RNN`: cálculo a mano, salidas y alcance de la memoria.
7. Modelo y entrenamiento con ponderación de clases.
8. Evaluación final en test.

### Sobre el conjunto de datos

Usamos **HDFS_v1**, de la colección [Loghub](https://github.com/logpai/loghub), publicado originalmente por [Xu et al. (2009)](https://doi.org/10.1145/1629575.1629587). Los logs se generaron al ejecutar trabajos de Hadoop en más de 200 nodos de Amazon EC2: unas 11 millones de líneas que corresponden a 575,061 bloques de archivos. Expertos en Hadoop etiquetaron cada bloque como normal o anómalo; 16,838 bloques (2.9%) son anómalos. Es uno de los datasets de referencia en detección de anomalías en logs, por ejemplo en [DeepLog (Du et al., 2017)](https://doi.org/10.1145/3133956.3134015).

In [ ]:
import ast
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from torchinfo import summary

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from utils import train, model_classification_report, plot_training

In [ ]:
# Fijamos la semilla para que los resultados sean reproducibles
SEED = 23

torch.manual_seed(SEED)
# cuDNN elige algoritmos deterministas (mismo resultado en cada corrida, algo más lento)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
from utils import get_device, get_num_workers

DEVICE = get_device()  # cuda > mps > xpu > cpu
# Linux: mitad de los núcleos disponibles, entre 1 y 8; Windows/macOS: 0 (ver docstring)
NUM_WORKERS = get_num_workers()

print(f"Usando {DEVICE}")
print(f"Usando {NUM_WORKERS} workers")

In [ ]:
BATCH_SIZE = 256

## Carga de datos

### Del log crudo a la secuencia de eventos

HDFS escribe una línea de log por cada evento. Estas son líneas reales del dataset:

```plaintext
081109 204815 653 INFO dfs.DataNode$DataXceiver: Receiving block blk_5792489080791696128 src: /10.251.30.6:33145 dest: /10.251.30.6:50010
081109 205035 28 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /user/root/rand/_temporary/_task_200811092030_0001_m_000590_0/part-00590. blk_-1727475099218615100
081109 204655 556 INFO dfs.DataNode$PacketResponder: Received block blk_3587508140051953248 of size 67108864 from /10.251.42.84
081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
081109 214043 2561 WARN dfs.DataNode$DataXceiver: 10.251.30.85:50010:Got exception while serving blk_-2918118818249673980 to /10.251.90.64:
```

Cada línea tiene fecha, hora, identificador de proceso, nivel, componente y un mensaje. El mensaje combina una parte fija, el **template** (`Receiving block <*> src: <*> dest: <*>`), con parámetros variables: direcciones IP, tamaños e identificadores de bloque. No usamos el texto directamente, sino un dataset ya preprocesado en tres pasos:

1. **Parsing**: un parser de logs (por ejemplo, Drain) extrae el template de cada línea y le asigna un **ID de evento** entero.
2. **Agrupación**: las líneas se agrupan por identificador de bloque (`blk_...`) y se ordenan en el tiempo. Cada bloque queda representado por la secuencia de IDs de sus eventos.
3. **Etiqueta**: cada bloque tiene la etiqueta normal o anómalo asignada por los expertos.

<img src="https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/hdfs_pipeline.png" width="900" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

El modelo no ve texto: ve secuencias de enteros, una por bloque.

In [ ]:
TRAIN_PATH = "data/log_classification_train.csv"
TEST_PATH = "data/log_classification_test.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

# La columna "sequence" está guardada como texto, "(4, 4, 21, ...)";
# ast.literal_eval la convierte en una tupla de enteros
train_df["events"] = train_df["sequence"].apply(ast.literal_eval)
test_df["events"] = test_df["sequence"].apply(ast.literal_eval)

print(f"Train: {len(train_df)} bloques | Test: {len(test_df)} bloques")

In [ ]:
train_df.head(10)

## Análisis exploratorio

Primero miramos la distribución de las clases y el largo de las secuencias.

In [ ]:
class_distribution = train_df["class"].value_counts()

plt.figure(figsize=(6, 4))
class_distribution.plot(kind="bar")
plt.title("Distribución de clases en el conjunto de entrenamiento")
plt.xlabel("Clase")
plt.ylabel("Frecuencia")
plt.show()

print((train_df["class"].value_counts(normalize=True) * 100).round(2))

Como en la clase 03 (MIT-BIH), el dataset está **desbalanceado**: menos del 3% de los bloques son anómalos. Un modelo que siempre predice "normal" obtiene más del 97% de accuracy y no detecta ninguna anomalía. Por eso vamos a mirar la precisión, el recall y el F1 de la clase anómala, y vamos a ponderar las clases en la pérdida.

In [ ]:
train_df["length"] = train_df["events"].apply(len)
test_df["length"] = test_df["events"].apply(len)

# density=True: cada histograma integra 1, así se comparan conjuntos de distinto tamaño
plt.figure(figsize=(10, 5))
plt.hist(train_df["length"], bins=range(0, 61), density=True, alpha=0.5, label="Train")
plt.hist(test_df["length"], bins=range(0, 61), density=True, alpha=0.5, label="Test")
plt.title("Distribución del largo de las secuencias (hasta 60 eventos)")
plt.xlabel("Largo de la secuencia")
plt.ylabel("Densidad")
plt.legend()
plt.show()

print(train_df["length"].describe(percentiles=[0.5, 0.9, 0.99]).round(1))

In [ ]:
plt.figure(figsize=(10, 5))
for cls in ["normal", "abnormal"]:
    lengths = train_df.loc[train_df["class"] == cls, "length"]
    plt.hist(lengths, bins=range(0, 61), density=True, alpha=0.5, label=cls)
plt.title("Distribución del largo de las secuencias por clase (hasta 60 eventos)")
plt.xlabel("Largo de la secuencia")
plt.ylabel("Densidad")
plt.legend()
plt.show()

train_df.groupby("class")["length"].agg(["min", "median", "max"])

Train y test tienen la misma distribución de largos. La mediana es de 19 eventos y el 99% de las secuencias tiene 32 o menos, aunque hay unas pocas de casi 300. Por clase aparece una diferencia: ningún bloque normal tiene menos de 13 eventos, mientras que hay bloques anómalos de solo 2. El largo ya es una pista para el modelo.

### ¿Qué valores tienen los datos?

In [ ]:
all_events = [e for seq in train_df["events"] for e in seq]

min_value = min(all_events)
max_value = max(all_events)

print(f"Valor mínimo: {min_value}")
print(f"Valor máximo: {max_value}")
print(f"Eventos distintos: {len(set(all_events))}")

In [ ]:
# Fracción de bloques de cada clase en los que aparece cada evento
event_sets = train_df["events"].apply(set)
event_ids = np.arange(min_value, max_value + 1)

plt.figure(figsize=(12, 4))
for offset, cls in [(-0.2, "normal"), (0.2, "abnormal")]:
    sets_cls = event_sets[train_df["class"] == cls]
    presence = [sets_cls.apply(lambda s: e in s).mean() for e in event_ids]
    plt.bar(event_ids + offset, presence, width=0.4, label=cls)
plt.title("Fracción de bloques en los que aparece cada evento")
plt.xlabel("ID de evento")
plt.ylabel("Fracción de bloques")
plt.xticks(event_ids)
plt.legend()
plt.show()

Dos conclusiones de esta exploración:

- **Los valores son categorías, no cantidades.** Hay 28 eventos distintos (IDs de 0 a 27). El evento 21 no es "mayor" que el evento 4: los números son nombres. Si los pasamos a la red como números, le imponemos un orden y unas distancias que no existen. Por eso vamos a representar cada evento con un vector aprendido (embedding).
- **Algunos eventos aparecen casi solo en bloques anómalos** (por ejemplo, 5, 15, 17, 19 y 24).

## Preprocesamiento de datos

Una RNN puede procesar secuencias de cualquier largo, pero para entrenar en batches necesitamos tensores rectangulares de forma `(batch, largo)`. Por eso fijamos un largo `SEQ_LEN`: las secuencias más largas se **truncan** y las más cortas se completan con un valor de relleno (**padding**).

El relleno necesita un valor que no sea un evento. Como los IDs empiezan en 0, sumamos 1 a todos (quedan de 1 a 28) y reservamos el **0 para el padding**.

Preguntas relevantes:
- ¿Qué largo de secuencia es adecuado para capturar la información relevante?
- ¿Qué es más importante, el principio o el final de la secuencia?
- ¿Conviene hacer el padding a la izquierda (*left padding*) o a la derecha (*right padding*)?

In [ ]:
SEQ_LEN = 50  # largo de las secuencias con las que vamos a trabajar
PADDING_VALUE = 0  # valor usado para el padding


def preprocess_sequence(
    sequence, max_len=SEQ_LEN, truncating="pre", padding="left", padding_value=PADDING_VALUE
):
    # Pasamos los IDs a base 1: el 0 queda libre para el padding
    sequence = [x + 1 for x in sequence]

    # Truncado: "pre" descarta el principio (conserva los últimos eventos), "post" descarta el final
    if len(sequence) > max_len:
        if truncating == "pre":
            sequence = sequence[-max_len:]
        else:
            sequence = sequence[:max_len]

    # Padding: "left" agrega los ceros antes de la secuencia, "right" después
    padding_size = max_len - len(sequence)
    if padding == "left":
        sequence = [padding_value] * padding_size + sequence
    else:
        sequence = sequence + [padding_value] * padding_size

    return sequence


train_df["padded_sequence"] = train_df["events"].apply(preprocess_sequence)
test_df["padded_sequence"] = test_df["events"].apply(preprocess_sequence)

print(f"Secuencias truncadas: {(train_df['length'] > SEQ_LEN).mean():.4%}")
print(f"Original: {train_df['events'][0]}")
print(f"Con padding: {train_df['padded_sequence'][0]}")

In [ ]:
# Porcentaje de secuencias que se truncarían con cada largo, por clase
for max_len in [30, 40, 50]:
    truncated = (train_df["length"] > max_len).groupby(train_df["class"]).mean() * 100
    print(f"SEQ_LEN = {max_len}: normal {truncated['normal']:.2f}% | abnormal {truncated['abnormal']:.2f}%")

Respuestas a las preguntas:

- **Largo.** El 99% de las secuencias tiene 32 eventos o menos, así que un largo de 30 parece suficiente. Sin embargo, las secuencias largas son sobre todo anómalas: con `SEQ_LEN = 30` se trunca el 3% de los bloques normales y casi el 10% de los anómalos, justo la clase que queremos detectar. Con `SEQ_LEN = 50` se trunca menos del 0.1% de cada clase. Conviene mirar el truncado por clase, y no solo sobre el total. Un largo mayor solo agregaría pasos de padding.
- **Principio o final.** Con `truncating="pre"` descartamos el principio y conservamos los últimos eventos, que son los que la RNN procesa justo antes de clasificar. En este dataset casi no hay truncado, pero en otros problemas es una decisión que depende de dónde está la información.
- **Izquierda o derecha.** Izquierda. El clasificador va a usar el último estado oculto de la RNN; con left padding, el último paso corresponde al último evento real. Lo vemos con un diagrama en la sección del modelo, y la comparación queda como ejercicio.

## Dataset y DataLoaders

In [ ]:
class LogDataset(Dataset):
    def __init__(self, sequences, labels):
        """
        Args:
            sequences (list of lists): secuencias preprocesadas (con padding), todas de largo SEQ_LEN.
            labels (list): etiquetas, 0 para normal y 1 para anómalo.
        """
        self.sequences = sequences
        self.labels = labels

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        # nn.Embedding y nn.CrossEntropyLoss esperan índices enteros (torch.long)
        sequence = torch.tensor(self.sequences[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return sequence, label

Separamos un conjunto de validación del conjunto de entrenamiento con `train_test_split(..., stratify=labels)`. El argumento `stratify` mantiene la misma proporción de anomalías en ambos conjuntos: con menos del 3% de anomalías, un split al azar puede dejar la validación con una proporción distinta. Como en las clases anteriores, comparamos los modelos en validación y usamos **test una sola vez**, al final.

In [ ]:
TARGET_NAMES = ["normal", "abnormal"]  # la etiqueta es el índice en esta lista

sequences = train_df["padded_sequence"].tolist()
labels = [TARGET_NAMES.index(cls) for cls in train_df["class"]]

train_seq, val_seq, train_labels, val_labels = train_test_split(
    sequences, labels, test_size=0.2, random_state=SEED, stratify=labels
)

train_dataset = LogDataset(train_seq, train_labels)
val_dataset = LogDataset(val_seq, val_labels)

test_seq = test_df["padded_sequence"].tolist()
test_labels = [TARGET_NAMES.index(cls) for cls in test_df["class"]]
test_dataset = LogDataset(test_seq, test_labels)

print(f"Train: {len(train_dataset)} | Validación: {len(val_dataset)} | Test: {len(test_dataset)}")

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

x, y = next(iter(train_loader))
print(f"Batch de entrada: {x.shape} {x.dtype}")  # (BATCH_SIZE, SEQ_LEN)
print(f"Batch de etiquetas: {y.shape} {y.dtype}")  # (BATCH_SIZE,)

## `nn.Embedding`

La [capa de embedding](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) es una **tabla de búsqueda**: una matriz de pesos de tamaño `num_embeddings × embedding_dim`, con una fila por evento. Dado un índice $i$, devuelve la fila $i$. Por ejemplo, con 100 símbolos y vectores de tamaño 20, la matriz es de 100 × 20.

Es equivalente a multiplicar el vector one-hot del evento por la matriz de pesos (una capa `Linear` sin bias), pero sin construir el one-hot. Las filas se aprenden con backpropagation como cualquier otro peso, así que eventos que cumplen un rol parecido en la tarea terminan con vectores parecidos.

¿Por qué no usar directamente el one-hot? Con 29 símbolos sería posible, pero su tamaño crece con el vocabulario (en texto hay decenas de miles de palabras) y no comparte información entre símbolos parecidos.

Para ver cómo se organizan los embeddings aprendidos sobre un vocabulario grande, el [Embedding Projector](https://projector.tensorflow.org/) permite explorar en 3D los vectores de word2vec y buscar los vecinos más cercanos de cada palabra. El video de StatQuest [*Word Embedding and Word2Vec, Clearly Explained!!!*](https://www.youtube.com/watch?v=viZrOnJclY0) muestra cómo una red aprende esos vectores.

Pregunta relevante:
- ¿Qué tamaño de embedding es adecuado? Reglas empíricas: la raíz cuadrada del tamaño del vocabulario ($\sqrt{29} \approx 5.4$) o su logaritmo en base 2 ($\log_2 29 \approx 4.9$). Usamos 8: con un vocabulario tan pequeño, el costo de algunas dimensiones extra es despreciable.

In [ ]:
# IDs originales 0..27 -> con el +1 quedan 1..28; más el 0 del padding: 29 filas (índices 0..28)
VOCAB_SIZE = max_value + 2
EMBEDDING_DIM = 8

embedding_layer = nn.Embedding(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    padding_idx=PADDING_VALUE,  # la fila del padding vale cero y no se actualiza
)
print(f"Matriz de pesos: {embedding_layer.weight.shape}")

word_indices = torch.tensor([0, 1, 2, 3, 4, 5], dtype=torch.long)  # índices enteros
embedded = embedding_layer(word_indices)

print(f"Índices: {word_indices}")
print(f"Embeddings {embedded.shape}:\n{embedded}")

La fila del índice 0 (padding) es un vector de ceros. Comprobamos la equivalencia con el producto entre el one-hot y la matriz de pesos:

In [ ]:
one_hot = F.one_hot(word_indices, num_classes=VOCAB_SIZE).float()  # (6, VOCAB_SIZE)
print(f"One-hot: {one_hot.shape}")
print(torch.allclose(one_hot @ embedding_layer.weight, embedded))

In [ ]:
word_indices = torch.tensor([2, 10, 50])
# embedding_layer(word_indices)  # ¿por qué falla?

> La tabla tiene filas para los índices 0 a 28: el índice 50 no existe y la capa lanza un `IndexError`. En GPU el mismo error aparece como un `device-side assert`, un mensaje más difícil de interpretar; conviene verificar el rango de los índices antes de entrenar.

## `nn.RNN`

Una red recurrente recorre la secuencia paso a paso. En cada paso $t$ combina la entrada $x_t$ con el estado oculto anterior $h_{t-1}$ y produce un nuevo estado $h_t$, un resumen de todo lo visto hasta $t$. La figura muestra la RNN **desenrollada en el tiempo**:

<!-- ![rnn](https://d2l.ai/_images/rnn.svg) -->
<img src="https://d2l.ai/_images/rnn.svg" width="500" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

Todas las cajas son **la misma capa con los mismos pesos**, aplicada en cada paso. Es la misma idea que el kernel de una convolución de la clase 04, que se desliza sobre la imagen: los pesos se comparten, así que la cantidad de parámetros no depende del largo de la secuencia.

La capa [`nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html) recibe un tensor `(batch, secuencia, características)` (con `batch_first=True`) y devuelve dos tensores:

- `output`: los estados ocultos de **todos los pasos**, `(batch, secuencia, hidden_size)`.
- `h_n`: el **último** estado oculto de cada capa, `(num_layers, batch, hidden_size)`.

Para clasificar una secuencia completa (un problema **many-to-one**: muchas entradas, una salida) basta con el último estado.

El video de StatQuest [*Recurrent Neural Networks (RNNs), Clearly Explained!!!*](https://www.youtube.com/watch?v=AsNTP8Kwu80) recorre estas ideas con animaciones: la red desenrollada, los pesos compartidos y el problema del gradiente.

**Fórmula matemática de la RNN**

Según la [documentación oficial de PyTorch](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html), para cada paso temporal $t$, una RNN calcula:

$$h_t = \tanh( x_t W_{ih}^T + b_{ih} +  h_{t-1} W_{hh}^T + b_{hh})$$

Donde:
- $x_t$ es la entrada en el tiempo $t$ (shape: `input_size`)
- $h_{t-1}$ es el estado oculto del paso anterior (shape: `hidden_size`)
- $h_t$ es el nuevo estado oculto (shape: `hidden_size`)
- $W_{ih}$ son los pesos input-to-hidden (shape: `hidden_size × input_size`)
- $W_{hh}$ son los pesos hidden-to-hidden (shape: `hidden_size × hidden_size`)
- $b_{ih}$ y $b_{hh}$ son los biases (shape: `hidden_size`)

$W_{ih}$, $W_{hh}$ y los biases son los mismos en todos los pasos. La $\tanh$ mantiene cada componente de $h_t$ en $[-1, 1]$; sin ella, aplicar $W_{hh}$ una y otra vez podría hacer crecer el estado sin límite.

> Notar que tiene dos términos de bias pero podría ser uno solo (PyTorch usa dos por compatibilidad con la implementación de cuDNN).

**Parámetros de la RNN**

Creamos una RNN muy pequeña para hacer las cuentas a mano.

In [ ]:
batch_size = 1  # un solo ejemplo
seq_len = 2  # secuencia de solo 2 pasos
input_size = 3  # cantidad de características de entrada
hidden_size = 4  # dimensión del hidden state

rnn = nn.RNN(
    input_size=input_size, hidden_size=hidden_size, batch_first=True
)  # por defecto batch_first=False

print("=== PARÁMETROS DE LA RNN ===\n")
for name, param in rnn.named_parameters():
    print(f"{name}:")
    print(f"  Shape: {param.shape}")
    print(f"  Valores:\n{param.data}\n")

In [ ]:
x = torch.tensor(
    [[[1.0, 2.0, 3.0], [0.5, 1.5, 2.5]]]  # paso temporal t=0  # paso temporal t=1
)  # (batch_size, seq_len, input_size)

# Extraer los parámetros
W_ih = rnn.weight_ih_l0.data  # (4, 3)
W_hh = rnn.weight_hh_l0.data  # (4, 4)
b_ih = rnn.bias_ih_l0.data  # (4,)
b_hh = rnn.bias_hh_l0.data  # (4,)

In [ ]:
# Notación: h_0 es el estado inicial (ceros por defecto); la entrada x_0 produce h_1, x_1 produce h_2
x_0 = x[:, 0, :]  # (1, 3) - primera entrada
h_0 = torch.zeros(batch_size, hidden_size)  # (1, 4)

print("=== PASO TEMPORAL t=0 ===\n")
print(f"x_0: {x_0}")
print(f"h_0 (inicial): {h_0}\n")

# Paso 1: contribución de la entrada
input_contrib = x_0 @ W_ih.T + b_ih  # (1, 4)
print(f"Input contribution (x_0 @ W_ih.T + b_ih):\n{input_contrib}\n")

# Paso 2: contribución del estado anterior
hidden_contrib = h_0 @ W_hh.T + b_hh  # (1, 4)
print(f"Hidden contribution (h_0 @ W_hh.T + b_hh):\n{hidden_contrib}\n")

# Paso 3: sumar y aplicar tanh
h_1 = torch.tanh(input_contrib + hidden_contrib)
print(f"h_1 = tanh(input_contrib + hidden_contrib):\n{h_1}\n")

In [ ]:
x_1 = x[:, 1, :]  # (1, 3) - segunda entrada; usamos h_1, el estado del paso anterior

print("=== PASO TEMPORAL t=1 ===\n")
print(f"x_1: {x_1}")
print(f"h_1 (anterior): {h_1}\n")

# Paso 1: contribución de la entrada (mismos W_ih y b_ih que en t=0)
input_contrib = x_1 @ W_ih.T + b_ih  # (1, 4)
print(f"Input contribution (x_1 @ W_ih.T + b_ih):\n{input_contrib}\n")

# Paso 2: contribución del estado anterior (mismos W_hh y b_hh que en t=0)
hidden_contrib = h_1 @ W_hh.T + b_hh  # (1, 4)
print(f"Hidden contribution (h_1 @ W_hh.T + b_hh):\n{hidden_contrib}\n")

# Paso 3: sumar y aplicar tanh
h_2 = torch.tanh(input_contrib + hidden_contrib)
print(f"h_2 = tanh(input_contrib + hidden_contrib):\n{h_2}\n")

Veamos si coincide con la implementación de PyTorch.

In [ ]:
output, hidden = rnn(x)

print("=== OUTPUT DE PYTORCH ===")
print(f"Output shape: {output.shape}")  # (batch_size, seq_len, hidden_size)
print(f"Output:\n{output}\n")

print(f"Hidden shape: {hidden.shape}")  # (num_layers, batch_size, hidden_size)
print(f"Hidden:\n{hidden}\n")

`output` contiene $h_1$ y $h_2$, los estados de los dos pasos. `hidden` contiene solo $h_2$, con una dimensión extra adelante: `nn.RNN` permite apilar varias capas recurrentes (`num_layers`) y devuelve el último estado de cada una. Con una sola capa, el último paso de `output` y `hidden[0]` son el mismo tensor:

In [ ]:
print(torch.allclose(output[:, -1], hidden[0]))  # último paso de output == último estado
print(torch.allclose(hidden[0], h_2))  # == nuestro cálculo a mano

Con las dimensiones de nuestro problema (un batch de secuencias de `SEQ_LEN` embeddings):

In [ ]:
rnn = nn.RNN(input_size=EMBEDDING_DIM, hidden_size=128, batch_first=True)
tensor = torch.randn(BATCH_SIZE, SEQ_LEN, EMBEDDING_DIM)

output, hidden = rnn(tensor)
print(f"Output shape: {output.shape} (batch_size, seq_len, hidden_size)")
print(f"Hidden shape: {hidden.shape} (num_layers, batch_size, hidden_size)")

### ¿Qué tan lejos llega la memoria de una RNN?

Durante el entrenamiento, el gradiente que llega a la entrada del paso $t$ atraviesa una multiplicación por $W_{hh}$ (y por la derivada de la $\tanh$) por cada paso que falta hasta el final. Si esos factores son menores que 1, el gradiente se achica de forma exponencial con la distancia: es el **vanishing gradient** de la clase 05. Una RNN desenrollada sobre $T$ pasos es una red de $T$ capas con pesos compartidos.

Lo medimos en una RNN sin entrenar: calculamos el gradiente del último estado $h_T$ respecto de cada entrada $x_t$.

In [ ]:
torch.manual_seed(SEED)
rnn_demo = nn.RNN(input_size=EMBEDDING_DIM, hidden_size=128, batch_first=True)

x = torch.randn(1, SEQ_LEN, EMBEDDING_DIM, requires_grad=True)
_, h_T = rnn_demo(x)
h_T.sum().backward()  # gradiente de (la suma de las componentes de) h_T respecto de cada x_t

grad_norm = x.grad[0].norm(dim=1)  # (SEQ_LEN,): norma del gradiente en cada paso
distance = SEQ_LEN - 1 - torch.arange(SEQ_LEN)  # distancia de cada paso al último

plt.figure(figsize=(8, 4))
plt.semilogy(distance, grad_norm, marker="o", markersize=3)
plt.title("Influencia de cada entrada en el último estado oculto")
plt.xlabel("Distancia al último paso (T - t)")
plt.ylabel(r"$\| \partial h_T / \partial x_t \|$ (escala log)")
plt.grid(True)
plt.show()

La escala del eje y es logarítmica: el último estado depende casi por completo de los últimos pasos, y la influencia de una entrada a 20 pasos de distancia es varios órdenes de magnitud menor. Dos consecuencias:

- Es otra razón para el **left padding**: los eventos reales quedan en los últimos pasos, los que más influyen en $h_T$.
- Una RNN simple aprende con dificultad dependencias entre eventos lejanos. Las arquitecturas **LSTM** y **GRU**, que vemos en la próxima clase, se diseñaron para atenuar este problema.

El artículo interactivo [*Visualizing memorization in RNNs*](https://distill.pub/2019/memorization-in-rnns/) (Distill) muestra el mismo fenómeno sobre texto: para cada predicción, colorea cuánto influye cada entrada anterior.

## Modelo

El clasificador encadena tres capas:

1. `nn.Embedding`: cada ID de evento se convierte en un vector de `EMBEDDING_DIM` componentes.
2. `nn.RNN`: recorre la secuencia de vectores y produce el último estado oculto $h_T$.
3. `nn.Linear`: transforma $h_T$ en dos logits (normal, anómala), que van a `nn.CrossEntropyLoss` como en la clase 03.

<img src="https://raw.githubusercontent.com/Catedra-IA/taller-deep-learning-letra/main/assets/rnn_many_to_one.png" width="900" style="background:white; display: block; margin-left: auto; margin-right: auto;"/>

En la fila de abajo se ve el problema del right padding: entre el último evento real y la salida, la RNN aplica varias veces la recurrencia sobre vectores nulos. Aunque la entrada sea cero, el estado sigue cambiando por $W_{hh}$ y los biases.

In [ ]:
# Salida esperada de summary (usarla para verificar la implementación):
#
# HDFSClassifier                           [256, 2]                  --
# ├─Embedding: 1-1                         [256, 50, 8]              232
# ├─RNN: 1-2                               [256, 50, 128]            17,664
# ├─Linear: 1-3                            [256, 2]                  258
# Total params: 18,154


class HDFSClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes=2):
        super(HDFSClassifier, self).__init__()
        # TODO: definir las capas como atributos:
        #   1. self.embedding: nn.Embedding de vocab_size a embedding_dim, con padding_idx=PADDING_VALUE
        #   2. self.rnn: nn.RNN de embedding_dim a hidden_dim, con batch_first=True
        #   3. self.fc: nn.Linear de hidden_dim a num_classes
        pass

    def forward(self, x):
        # x: (batch, SEQ_LEN) con índices enteros
        # TODO:
        #   1. embedding -> (batch, SEQ_LEN, embedding_dim)
        #   2. rnn -> output, hidden; hidden tiene forma (num_layers, batch, hidden_dim)
        #   3. último estado de la última capa: hidden[-1] -> (batch, hidden_dim)
        #   4. fc -> logits (batch, num_classes); sin activación (CrossEntropyLoss incluye la softmax)
        pass


HIDDEN_DIM = 128

summary(
    HDFSClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM),
    input_size=(BATCH_SIZE, SEQ_LEN),
    dtypes=[torch.long],
)

El modelo tiene unos 18 mil parámetros, y casi todos están en la RNN: $W_{ih}$ (128 × 8), $W_{hh}$ (128 × 128) y los dos biases. Ninguno depende de `SEQ_LEN`.

## Entrenamiento

Primero entrenamos con `nn.CrossEntropyLoss()` sin pesos, como en la clase 03.

In [ ]:
CRITERION = nn.CrossEntropyLoss()
LR = 5e-4
EPOCHS = 20
PATIENCE = 5

In [ ]:
torch.manual_seed(SEED)  # inicialización reproducible (la reutilizamos al entrenar con pesos)
model = HDFSClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

In [ ]:
train_errors, val_errors = train(
    model,
    optimizer=optimizer,
    criterion=CRITERION,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=PATIENCE,
    epochs=EPOCHS,
)

In [ ]:
plot_training(train_errors, val_errors, mark_min=True)

In [ ]:
model_classification_report(
    model, val_loader, DEVICE, 2, digits=4, target_names=TARGET_NAMES, do_confusion_matrix=True
)

La accuracy es casi perfecta, pero con un 3% de anomalías esa métrica dice poco. Lo que importa es la fila de la clase anómala en la matriz de confusión (filas: clase real; columnas: predicción): cada falso negativo es una anomalía que el modelo deja pasar. ¿Podemos entrenar al modelo para que priorice no perder anomalías?

### Ponderación de clases

En la clase 03 mencionamos que una forma de tratar el desbalance es dar más peso a la clase minoritaria en la pérdida. Con `nn.CrossEntropyLoss(weight=...)`, la pérdida de cada ejemplo se multiplica por el peso de su clase. Con pesos inversamente proporcionales a la frecuencia ($\text{peso}_c = N / N_c$), equivocarse en una anomalía cuesta unas 34 veces más que equivocarse en un bloque normal, y las dos clases aportan lo mismo a la pérdida total.

In [ ]:
class_counts = Counter(train_labels)
total = sum(class_counts.values())
class_weights = torch.tensor(
    [total / class_counts[0], total / class_counts[1]], dtype=torch.float
).to(DEVICE)  # en el mismo dispositivo que los logits

print(f"Cantidad por clase: {dict(class_counts)}")
print(f"Pesos: {class_weights}")

Veamos el efecto sobre dos ejemplos que el modelo clasifica como normales (el logit de "normal" es mayor): el primero es efectivamente normal y el segundo es una anomalía. Con `reduction="none"` obtenemos la pérdida de cada ejemplo por separado.

In [ ]:
logits = torch.tensor([[2.0, -1.0], [2.0, -1.0]])  # ambos ejemplos se predicen como "normal"
targets = torch.tensor([0, 1])  # el primero es normal y el segundo anómalo

loss_plain = F.cross_entropy(logits, targets, reduction="none")
loss_weighted = F.cross_entropy(logits, targets, weight=class_weights.cpu(), reduction="none")

print(f"Sin pesos:  normal = {loss_plain[0]:.3f} | anómalo (error) = {loss_plain[1]:.3f}")
print(f"Con pesos:  normal = {loss_weighted[0]:.3f} | anómalo (error) = {loss_weighted[1]:.3f}")

Sin pesos, el error sobre la anomalía cuesta lo mismo que el de cualquier otro ejemplo; como las anomalías son menos del 3% del batch, aportan poco al gradiente. Con pesos, ese mismo error cuesta unas 34 veces más.

<small>Con pesos, `reduction="mean"` (el valor por defecto) divide por la suma de los pesos de los ejemplos del batch y no por la cantidad de ejemplos. Por eso los valores de una pérdida ponderada no son comparables con los de una pérdida sin ponderar.</small>

Entrenamos el mismo modelo, con la misma inicialización, usando la pérdida ponderada:

In [ ]:
CRITERION_WEIGHTED = nn.CrossEntropyLoss(weight=class_weights)  # pérdida ponderada por clase

torch.manual_seed(SEED)  # misma inicialización que el modelo sin pesos
model_weighted = HDFSClassifier(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM).to(DEVICE)
optimizer_weighted = optim.Adam(model_weighted.parameters(), lr=LR)

train_errors_weighted, val_errors_weighted = train(
    model_weighted,
    optimizer=optimizer_weighted,
    criterion=CRITERION_WEIGHTED,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    do_early_stopping=True,
    patience=PATIENCE,
    epochs=EPOCHS,
)

In [ ]:
plot_training(train_errors_weighted, val_errors_weighted, mark_min=True)

La pérdida de validación del modelo ponderado tiene picos más grandes. La validación tiene unas 2,250 anomalías, y cada error sobre ellas pesa unas 34 veces más que un error sobre un bloque normal: pocos errores bastan para producir un pico. Con early stopping, el modelo queda con los pesos de la época de menor pérdida de validación.

In [ ]:
model_classification_report(
    model_weighted, val_loader, DEVICE, 2, digits=4, target_names=TARGET_NAMES, do_confusion_matrix=True
)

### Comparación

En nuestra ejecución, en validación:

| | Falsos negativos (anomalías no detectadas) | Falsos positivos (falsas alarmas) | F1 de la clase anómala |
|---|---|---|---|
| Sin pesos | 12 | 23 | 0.992 |
| Con pesos | 2 | 101 | 0.978 |

La ponderación **no mejora el F1** (incluso lo baja): desplaza el compromiso entre precisión y recall. El modelo ponderado deja pasar menos anomalías a cambio de más falsas alarmas.

Cuál conviene depende del costo de cada error en la aplicación. Si una anomalía no detectada significa una falla en producción, detectar 10 anomalías más puede justificar unas 80 revisiones adicionales. También vemos que, con un 3% de anomalías y una señal tan clara, el modelo sin pesos no colapsa a predecir siempre "normal": la ponderación se vuelve imprescindible cuando la clase minoritaria es mucho más rara o más difícil de separar.

## Evaluación final en test

En esta aplicación priorizamos no dejar pasar anomalías, así que elegimos el modelo ponderado y lo evaluamos una sola vez sobre el conjunto de test.

In [ ]:
model_classification_report(
    model_weighted,
    test_loader,
    DEVICE,
    2,
    digits=4,
    target_names=TARGET_NAMES,
    do_confusion_matrix=True,
    do_balanced_accuracy=True,
)

En la matriz de confusión (filas: clase real; columnas: predicción) los dos tipos de error tienen costos distintos en operación:

- **Falso positivo** (un bloque normal marcado como anómalo): un operador revisa un bloque sin problemas.
- **Falso negativo** (una anomalía no detectada): una falla pasa desapercibida.

Con un peso de 34 para la clase anómala, el modelo prefiere cometer falsos positivos antes que falsos negativos: el recall de las anomalías es muy alto y la precisión es algo menor. Los pesos de la pérdida son una forma de ajustar ese compromiso.

## Conclusiones

- Un log se convierte en datos para una red en tres pasos: parsing a templates, agrupación por bloque y codificación de cada evento con un entero.
- Los IDs de evento son categorías: `nn.Embedding` les asigna un vector aprendido, equivalente a una capa lineal sobre el one-hot.
- Una RNN aplica la misma capa en cada paso y resume la secuencia en el estado oculto. `output` contiene todos los estados y `h_n` el último de cada capa.
- La influencia de una entrada sobre el último estado decae de forma exponencial con la distancia (vanishing gradient). El left padding deja los eventos reales en los últimos pasos.
- Con ponderación de clases, el modelo detecta casi todas las anomalías del test, a costa de algunos falsos positivos.

## Ejercicios

1. **Right padding**: cambiar a `padding="right"` en `preprocess_sequence`, reentrenar y comparar en validación. Relacionar el resultado con el diagrama del modelo y con el gráfico de la memoria de la RNN.
2. **Pesos intermedios**: probar pesos más suaves, por ejemplo la raíz cuadrada de $N / N_c$ o `[1, 5]`. ¿Cómo se mueven la precisión y el recall de la clase anómala entre el modelo sin pesos y el ponderado?
3. **Largo de secuencia**: probar `SEQ_LEN = 20` y `SEQ_LEN = 30` y comparar en validación, en particular el recall de las anomalías largas. ¿Cuánto se pierde al truncar? Recordar que `truncating="pre"` conserva los últimos eventos.
4. **GRU y LSTM**: reemplazar `nn.RNN` por `nn.GRU` (misma interfaz) y por `nn.LSTM` (devuelve `output, (h_n, c_n)`). ¿Qué cambia en la cantidad de parámetros y en el resultado?
5. **RNN apilada**: usar `num_layers=2` y `dropout=0.2` en `nn.RNN`. ¿Qué forma tiene `hidden`? ¿Por qué el `forward` usa `hidden[-1]`?

#### Lectura adicional

- Xu et al. (2009), [*Detecting Large-Scale System Problems by Mining Console Logs*](https://doi.org/10.1145/1629575.1629587): el trabajo que publicó el dataset de HDFS.
- Du et al. (2017), [*DeepLog: Anomaly Detection and Diagnosis from System Logs through Deep Learning*](https://doi.org/10.1145/3133956.3134015): detección de anomalías en logs con LSTM.
- Zhu et al. (2023), [*Loghub: A Large Collection of System Log Datasets for AI-driven Log Analytics*](https://arxiv.org/abs/2008.06448), y el [repositorio de Loghub](https://github.com/logpai/loghub).
- Andrej Karpathy (2015), [*The Unreasonable Effectiveness of Recurrent Neural Networks*](http://karpathy.github.io/2015/05/21/rnn-effectiveness/): qué pueden aprender las RNN, con ejemplos.
- StatQuest, [*Recurrent Neural Networks (RNNs), Clearly Explained!!!*](https://www.youtube.com/watch?v=AsNTP8Kwu80) (video): la RNN desenrollada, los pesos compartidos y el vanishing gradient, con animaciones.
- The AI Hacker, [*Illustrated Guide to Recurrent Neural Networks*](https://www.youtube.com/watch?v=LHXXI4-IEns) (video): cómo el estado oculto transporta información de un paso al siguiente.
- Distill (2019), [*Visualizing memorization in RNNs*](https://distill.pub/2019/memorization-in-rnns/) (interactivo): cuánto influye cada entrada anterior en cada predicción.
- Chris Olah (2015), [*Understanding LSTM Networks*](https://colah.github.io/posts/2015-08-Understanding-LSTMs/): diagramas de la RNN desenrollada, el problema de las dependencias largas y la LSTM.
- [Embedding Projector](https://projector.tensorflow.org/) (interactivo): exploración en 3D de los embeddings de word2vec.
- StatQuest, [*Word Embedding and Word2Vec, Clearly Explained!!!*](https://www.youtube.com/watch?v=viZrOnJclY0) (video): cómo una red aprende los embeddings.
- Jay Alammar, [*The Illustrated Word2vec*](https://jalammar.github.io/illustrated-word2vec/): introducción ilustrada a la idea de representar cada elemento con un vector.
- 3Blue1Brown, [*Transformers, the tech behind LLMs*](https://www.youtube.com/watch?v=wjZofJX0v4M) (video): la primera parte explica la matriz de embeddings como tabla de búsqueda y las direcciones con significado en ese espacio.
- Pascanu et al. (2013), [*On the difficulty of training Recurrent Neural Networks*](https://arxiv.org/abs/1211.5063): vanishing y exploding gradients en RNN.
- [Dive into Deep Learning, capítulo 9](https://d2l.ai/chapter_recurrent-neural-networks/index.html): redes recurrentes.
- Documentación de PyTorch: [`nn.Embedding`](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html) y [`nn.RNN`](https://pytorch.org/docs/stable/generated/torch.nn.RNN.html).